### 제목과 본문 미리보기  (NaverBlog-Twitter-Youtube)

In [12]:
from selenium import webdriver
import time
import pandas as pd


def crawling():
    data = []
    BLOG_COUNT_PER_PAGE = 7  # 한 페이지당 글 7개
    
    keyword = input('Keyword : ')  # 키워드 입력
    try:
        page_num = int(input('Page Num : '))  # 총 페이지 수 입력
    except:
        print('Please press valid page number (int)')
        sys.exit(1)

    driver = webdriver.Chrome('./chromedriver.exe')
    driver.maximize_window()

    
    temporary_storage_num = 1
    for PAGE_NUM in range(1, page_num + 1):
        print(f'Start crawling page {PAGE_NUM}')
        
        NAVER_URL = f'https://section.blog.naver.com/Search/Post.nhn?pageNo={PAGE_NUM}&rangeType=ALL&orderBy=sim&keyword={keyword}'
        driver.get(NAVER_URL)  # page 번호에 해당하는 url에 접속
        
        # page 로딩 시간 기다림
        time.sleep(1.5)
        for BLOG_NUM in range(1, BLOG_COUNT_PER_PAGE + 1):
            try:
                # 블로그 제목에 해당하는 css selector
                title_selector = f'#content > section > div.area_list_search > div:nth-child({BLOG_NUM}) > div > div.info_post > div > a.desc_inner > strong > span.title'
                
                # 블로그 미리보기 내용에 해당하는 css selector
                content_selector = f'#content > section > div.area_list_search > div:nth-child({BLOG_NUM}) > div > div.info_post > div > a.text'

                # 제목과 내용을 가져옴
                title  = driver.find_element_by_css_selector(title_selector).text
                content = driver.find_element_by_css_selector(content_selector).text

                # 특수기호 없애는 작업
                for idx in range(len(title)):
                    if not ((0 <= ord(title[idx]) < 128) or (0xac00 <= ord(title[idx]) <= 0xd7af)):
                        title = title.replace(title[idx], ' ')
                for idx in range(len(content)):
                    if not ((0 <= ord(content[idx]) < 128) or (0xac00 <= ord(content[idx]) <= 0xd7af)):
                        content = content.replace(content[idx], ' ')

                data.append([title, content])
                
            except Exception as e:
                pass

        # 컴퓨터 메모리 때문에 50 page씩 데이터를 저장하고 버퍼를 지운다.
        if temporary_storage_num % 50 == 0:
            dataframe = pd.DataFrame(data, columns=["title", "content"])
            dataframe.to_csv('./naver_comment.csv', mode='a', encoding='cp949')
            data = []
            
        temporary_storage_num += 1
  
    driver.close()       
    print('Finish crawling')

    dataframe = pd.DataFrame(data, columns=["title", "content"])
    dataframe.to_csv('./naver_comment.csv', mode='a', encoding='cp949')       
    print('All done!')


In [13]:
crawling()

Keyword : 강아지
Page Num : 3
Start crawling page 1
Start crawling page 2
Start crawling page 3
Finish crawling
All done!


### 네이버 API 사용

In [ ]:
import os
import sys
import urllib.request
import datetime
import time
import json
from config import *
import time

#[CODE 1]
def get_request_url(url):
    
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id", 'l_7jUWDW3SVxyKz5KJfi')
    req.add_header("X-Naver-Client-Secret", "pL9UpsRpkh")
    time.sleep(1)
    try: 
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print ("[%s] Url Request Success" % datetime.datetime.now())
            print (url)
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for URL : %s" % (datetime.datetime.now(), url))
        return None

#[CODE 2]
def getNaverSearchResult(sNode, search_text, page_start, display):
    
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % sNode
    parameters = "?query=%s&start=%s&display=%s" % (urllib.parse.quote(search_text), page_start, display)
    url = base + node + parameters
    
    retData = get_request_url(url)
    
    if (retData == None):
        return None
    else:
        return json.loads(retData)

#[CODE 3]
def getPostData(post, jsonResult):
    
    title = post['title']
    description = post['description']
    org_link = post['bloggerlink']
    link = post['link']
    pDate = post['postdate']

#     #Tue, 14 Feb 2017 18:46:00 +0900
#     pDate = datetime.datetime.strptime(post['postdate'],  '%a, %d %b %Y %H:%M:%S +0900')
#     pDate = pDate.strftime('%Y-%m-%d %H:%M:%S')
    
    jsonResult.append({'title':title, 'description': description,
                    'org_link':org_link, 'link': org_link, 
                    'postdate':pDate})
    return    


In [ ]:
def main():

    jsonResult = []

    # 'news', 'blog', 'cafearticle'
    sNode = 'blog'
#     search_text = ["유한양행","SK바이오팜","씨젠","오스템임플란트","차바이오텍","헬릭스미스","메지온","에이치엘비생명과학","대웅","녹십자홀딩스",
# "동국제약","CMG제약","영진약품","엘앤씨바이오","메디포스트","제테마","코미팜","엔케이맥스","바디텍메드","JW중외제약","클래시스","종근당홀딩스",
# "유틸렉스","서흥","티움바이오","카이노스메드","앱클론","아이센스","EDGC","에이프로젠제약","레이","바텍","삼진제약","동화약품","수젠텍","제일약품",
# "뷰웍스","테라젠이텍스","한독","케어젠","대원제약","유비케어","일동제약","동성제약","지노믹트리","제일파마홀딩스"]
    search_text = ["강아지"]
    display_count = 10 # 기사 100개씩 저장
    
    for i in search_text:
        jsonResult = []
        jsonSearch = getNaverSearchResult(sNode, i, 1, display_count)
        
        while ((jsonSearch != None) and (jsonSearch['display'] != 0)):
            for post in jsonSearch['items']:
                print(post)
                getPostData(post, jsonResult)
            
            nStart = jsonSearch['start'] + jsonSearch['display']
            jsonSearch = getNaverSearchResult(sNode, i, nStart, display_count)
        
        with open('%s_naver_%s.json' % (i, sNode), 'w', encoding='utf8') as outfile:
            retJson = json.dumps(jsonResult,
                            indent=4, sort_keys=True,
                            ensure_ascii=False)
            outfile.write(retJson)
            
        print ('%s_naver_%s.json SAVED' % (i, sNode))

    
if __name__ == '__main__':
    main()